# Worker A: Healthcare (Full Scale)

Runs the full Obermeyer healthcare grid. Uses analytic (KKT-based) decision gradients.

**Grid:** 7 methods × 2 alphas × 5 seeds = 70 runs

Checkpointed: re-run cells to resume from where you left off.

In [ ]:
import os, sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ----- Path setup -----
DRIVE_ROOT = '/content/drive/MyDrive/DecisionFocusedMTL'
MOSEK_LIC_PATH = f'{DRIVE_ROOT}/data/mosek.lic'

if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH
    print(f'MOSEK license set: {MOSEK_LIC_PATH}')
else:
    print(f'Warning: MOSEK license not found at {MOSEK_LIC_PATH}')

for p in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

%cd {DRIVE_ROOT}

# Results directory
HC_RESULTS = os.path.join(DRIVE_ROOT, 'results', 'final', 'healthcare')
os.makedirs(HC_RESULTS, exist_ok=True)

from experiments.colab_runner import *
import torch
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Results: {HC_RESULTS}')

## Configuration

In [ ]:
# ===== TASK PARAMETERS =====
TASK_OVERRIDES = {
    'n_sample': 0,                 # 0 = all 48,784 patients (full scale)
    'val_fraction': 0.1,           # 10% validation split
    # 'budget_rho': 0.35,
    # 'test_fraction': 0.5,
    # 'decision_mode': 'group',
    # 'fairness_type': 'mad',
}

# ===== TRAINING PARAMETERS =====
TRAIN_OVERRIDES = {
    # Optimizer
    'optimizer': 'adamw',           # 'sgd', 'sgd_momentum', 'adam', 'adamw'
    'lr': 0.001,                    # Learning rate
    'weight_decay': 1e-4,           # Decoupled weight decay (adamw)
    'lr_warmup_steps': 5,           # Linear LR warmup
    # Model architecture
    'init_mode': 'best_practice',   # Kaiming He initialization
    'dropout': 0.1,                 # Dropout rate
    'hidden_dim': 64,               # MLP hidden width
    'n_layers': 2,                  # MLP depth
}

# ===== EXPERIMENT CONTROL =====
STEPS = 70
OVERWRITE = False

print('='*60)
print('HEALTHCARE EXPERIMENT — FULL SCALE')
print('='*60)
print(f'Steps per lambda: {STEPS}')
print(f'Overwrite: {OVERWRITE}')
print(f'Task overrides: {TASK_OVERRIDES}')
print(f'Train overrides: {TRAIN_OVERRIDES}')
print(f'Results dir: {HC_RESULTS}')
print('='*60)

## Run All Methods

In [ ]:
results = run_healthcare_slice(
    results_dir=HC_RESULTS,
    steps=STEPS,
    task_overrides=TASK_OVERRIDES,
    train_overrides=TRAIN_OVERRIDES,
    overwrite=OVERWRITE,
)

## Results

In [ ]:
show_progress(HC_RESULTS, 'Healthcare — COMPLETE')

if not results.empty:
    cols = ['test_regret_normalized', 'test_fairness', 'test_pred_mse']
    for alpha in sorted(results['alpha_fair'].unique()):
        sub = results[results['alpha_fair'] == alpha]
        print(f'\n{"="*60}')
        print(f'alpha = {alpha} (mean +/- std over seeds)')
        print(f'{"="*60}')
        summary = sub.groupby(['method_label', 'lambda'])[cols].agg(['mean', 'std']).round(4)
        print(summary.to_string())

        # SAA comparison
        saa = sub[sub['method_label'] == 'SAA']
        if not saa.empty:
            saa_reg = saa['test_regret_normalized'].mean()
            saa_mse = saa['test_pred_mse'].mean()
            print(f'\nSAA baseline: Regret={saa_reg:.4f}, MSE={saa_mse:.1f}')
            for m in ['FPTO', 'WDRO', 'FDFL-Scal', 'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']:
                ms = sub[(sub['method_label'] == m) & (sub['lambda'] == 0.0)]
                if not ms.empty:
                    r = ms['test_regret_normalized'].mean()
                    print(f'  {m:15s}: Regret={r:.4f} ({(r-saa_reg)/saa_reg*100:+.1f}% vs SAA)')

Worker A complete. Run the Results notebook to analyze.